# Mulberry Leaf Defect Detection - Optimized Pipeline
This notebook downloads the dataset, applies data augmentation, trains a MobileNetV2 model with EarlyStopping and Checkpoints, evaluates the results (Confusion Matrix & Classification Report), and saves the model to `.h5`, `.keras`, and `.tflite`.

In [1]:
!pip install -q gdown scikit-learn seaborn matplotlib opencv-python
import os
import gdown
import zipfile
import tensorflow as tf
from tensorflow.keras.preprocessing.image import ImageDataGenerator
from tensorflow.keras.callbacks import EarlyStopping, ModelCheckpoint, ReduceLROnPlateau
import matplotlib.pyplot as plt
import pandas as pd
import numpy as np
from sklearn.metrics import classification_report, confusion_matrix
import seaborn as sns

## 1. Download and Prepare Dataset

In [4]:
# folder_id = "10OmBK7HqUxqMNALnBBZg1uHkD06NGDIY"
output_dir = "Mulberry dataset"

# Downloading dataset from Google Drive 
# if not os.path.exists(output_dir):
#     gdown.download_folder(id=folder_id, output=output_dir, quiet=False, use_cookies=False)

train_dir = None
val_dir = None
test_dir = None

for root, dirs, files in os.walk(output_dir):
    if 'train' in dirs and 'val' in dirs and 'test' in dirs:
        train_dir = os.path.join(root, 'train')
        val_dir = os.path.join(root, 'val')
        test_dir = os.path.join(root, 'test')
        break

if train_dir is None:
    base_dir = "Mulberry dataset"
    if os.path.exists(base_dir):
        train_dir = f"{base_dir}/train"
        val_dir = f"{base_dir}/val"
        test_dir = f"{base_dir}/test"
    else:
        raise Exception("Dataset directories not found!")

print(f"Train: {train_dir}\nVal: {val_dir}\nTest: {test_dir}")

Train: Mulberry dataset\train
Val: Mulberry dataset\val
Test: Mulberry dataset\test


## 2. Data Augmentation and Generators

In [5]:
print("Setting up Data Generators...")
datagen_train = ImageDataGenerator(
    rescale=1./255,
    rotation_range=30,
    width_shift_range=0.2,
    height_shift_range=0.2,
    shear_range=0.2,
    zoom_range=0.2,
    horizontal_flip=True,
    vertical_flip=True,
    fill_mode='nearest'
)

datagen_val_test = ImageDataGenerator(rescale=1./255)

train_generator = datagen_train.flow_from_directory(
    directory=train_dir,
    target_size=(224, 224),
    keep_aspect_ratio=True,
    batch_size=32,
    class_mode='sparse'
)

val_generator = datagen_val_test.flow_from_directory(
    directory=val_dir,
    target_size=(224, 224),
    # keep_aspect_ratio=True,
    batch_size=32,
    class_mode='sparse'
)

test_generator = datagen_val_test.flow_from_directory(
    directory=test_dir,
    target_size=(224, 224),
    # keep_aspect_ratio=True,
    batch_size=32,
    class_mode='sparse',
    shuffle=False
)

Setting up Data Generators...
Found 1357 images belonging to 3 classes.
Found 192 images belonging to 3 classes.
Found 392 images belonging to 3 classes.


## 3. Build Model (MobileNetV2)

In [ ]:
print("Building MobileNetV2 Model...")
base_model = tf.keras.applications.MobileNetV2(
    input_shape=(224, 224, 3),
    include_top=False,
    weights='imagenet',
    pooling='max'
)
base_model.trainable = False  # Freeze base model

inputs = base_model.input
outputs = tf.keras.layers.Dense(train_generator.num_classes, activation='softmax')(base_model.output)
model = tf.keras.Model(inputs, outputs)

model.compile(
    optimizer=tf.keras.optimizers.Adam(learning_rate=1e-4),
    loss='sparse_categorical_crossentropy',
    metrics=['accuracy']
)
model.summary()

## 4. Training with Callbacks

In [ ]:
callbacks = [
    ModelCheckpoint(
        filepath='best_model.keras',
        monitor='val_accuracy',
        save_best_only=True,
        mode='max',
        verbose=1
    ),
    EarlyStopping(
        monitor='val_accuracy',
        patience=5,
        restore_best_weights=True,
        verbose=1
    ),
    ReduceLROnPlateau(
        monitor='val_loss',
        factor=0.2,
        patience=3,
        min_lr=1e-6,
        verbose=1
    )
]

print("Starting Training...")
history = model.fit(
    train_generator,
    validation_data=val_generator,
    epochs=15,
    callbacks=callbacks
)

12/43 ━━━━━━━━━━━━━━━━━━━━ 31s 1s/step - accuracy: 0.6081 - loss: 1.3107   

## 5. Evaluation and Metrics

In [ ]:
print("Evaluating on Test Set...")
results = model.evaluate(test_generator, verbose=1)
print(f"Test Loss: {results[0]:.5f}")
print(f"Test Accuracy: {results[1]*100:.2f}%")

pred = model.predict(test_generator)
pred_classes = np.argmax(pred, axis=1)
true_classes = test_generator.classes

labels_dict = train_generator.class_indices
class_names = [k for k, v in sorted(labels_dict.items(), key=lambda item: item[1])]

report = classification_report(true_classes, pred_classes, target_names=class_names, digits=4)
print("\nClassification Report:\n", report)

cf_matrix = confusion_matrix(true_classes, pred_classes)
plt.figure(figsize=(8, 6))
sns.heatmap(cf_matrix, annot=True, fmt='d', cmap="Blues",
            xticklabels=class_names, yticklabels=class_names)
plt.xlabel("Predicted")
plt.ylabel("True")
plt.title("Confusion Matrix")
plt.show()

plt.figure(figsize=(12, 4))
plt.subplot(1, 2, 1)
plt.plot(history.history['accuracy'], label='train_acc')
plt.plot(history.history['val_accuracy'], label='val_acc')
plt.title('Accuracy')
plt.legend()

plt.subplot(1, 2, 2)
plt.plot(history.history['loss'], label='train_loss')
plt.plot(history.history['val_loss'], label='val_loss')
plt.title('Loss')
plt.legend()
plt.show()

## 6. Save Models (.h5 and .tflite)

In [ ]:
model.save("mulberry_leaf_classification.h5")
print("Model saved as mulberry_leaf_classification.h5")

print("Converting to TFLite format...")
converter = tf.lite.TFLiteConverter.from_keras_model(model)
converter.optimizations = [tf.lite.Optimize.DEFAULT]
tflite_model = converter.convert()

with open('mulberry_leaf_classification.tflite', 'wb') as f:
    f.write(tflite_model)
print("Model successfully converted and saved as mulberry_leaf_classification.tflite")